# Web scraping Doctolib avec Selenium

Objectif : rechercher les chirurgiens-dentistes à Paris, ouvrir leurs fiches et récupérer leur nom, leur adresse et leur présentation complète.

> Les sélecteurs d'un site web peuvent évoluer. Ce notebook utilise plusieurs sélecteurs de secours et une limite configurable pour faciliter les tests.

## Étape 1 - Installer les bibliothèques

Décommenter la ligne suivante uniquement si les bibliothèques ne sont pas déjà installées.

In [2]:
%pip install selenium pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Étape 2 - Importer les bibliothèques et préparer les fonctions utiles

In [3]:
import time
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

TIMEOUT = 15
MAX_DENTISTES = 20  # Mettre None pour traiter tous les profils trouvés.


def attendre_et_cliquer(driver, selecteurs, timeout=TIMEOUT):
    """Clique sur le premier élément disponible parmi plusieurs sélecteurs."""
    for by, valeur in selecteurs:
        try:
            element = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((by, valeur))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
            element.click()
            return element
        except (TimeoutException, NoSuchElementException):
            continue
    return None


def trouver_premier(driver, selecteurs):
    """Renvoie le premier élément présent parmi plusieurs sélecteurs."""
    for by, valeur in selecteurs:
        elements = driver.find_elements(by, valeur)
        if elements:
            return elements[0]
    return None


def texte_premier(driver, selecteurs, valeur_par_defaut=""):
    element = trouver_premier(driver, selecteurs)
    return element.text.strip() if element else valeur_par_defaut

## Étape 3 - Ouvrir Doctolib et refuser les cookies

In [4]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

# Selenium Manager télécharge automatiquement le pilote adapté si nécessaire.
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, TIMEOUT)

driver.get("https://www.doctolib.fr/")

bouton_refus = attendre_et_cliquer(driver, [
    (By.ID, "didomi-notice-disagree-button"),
    (By.XPATH, "//button[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'refuser')]")
], timeout=5)

print("Cookies refusés." if bouton_refus else "Aucun bandeau de cookies affiché.")

NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


## Étape 4 - Rechercher un chirurgien-dentiste à Paris

In [ ]:
# Champ spécialité
champ_specialite = wait.until(EC.element_to_be_clickable((
    By.XPATH,
    "//input[contains(@placeholder, 'spécialité') or contains(@placeholder, 'nom') or contains(@aria-label, 'profession')]"
)))
champ_specialite.clear()
champ_specialite.send_keys("dentiste")

attendre_et_cliquer(driver, [
    (By.XPATH, "//*[self::div or self::li or self::button][contains(normalize-space(.), 'Chirurgien-dentiste')]")
])

# Champ région ou lieu
champ_lieu = wait.until(EC.element_to_be_clickable((
    By.XPATH,
    "//input[contains(@placeholder, 'Où') or contains(@placeholder, 'lieu') or contains(@placeholder, 'ville') or contains(@aria-label, 'lieu')]"
)))
champ_lieu.clear()
champ_lieu.send_keys("Paris")

attendre_et_cliquer(driver, [
    (By.XPATH, "//*[self::div or self::li or self::button][contains(normalize-space(.), 'Paris')]")
])

attendre_et_cliquer(driver, [
    (By.XPATH, "//button[@type='submit']"),
    (By.XPATH, "//button[contains(normalize-space(.), 'Rechercher')]")
])

wait.until(lambda navigateur: "dentiste" in navigateur.current_url.lower())
print("Page de résultats :", driver.current_url)

## Étape 5 - Récupérer les liens des fiches de dentistes

Le défilement permet de charger davantage de résultats. La limite `MAX_DENTISTES` définie à l'étape 2 accélère les essais.

In [ ]:
def recuperer_liens_profils(driver, maximum=None, nombre_defilements=15):
    liens = set()
    hauteur_precedente = 0

    for _ in range(nombre_defilements):
        for element in driver.find_elements(By.CSS_SELECTOR, "a[href]"):
            href = element.get_attribute("href") or ""
            # Les fiches de praticiens contiennent généralement /dentiste/ dans leur URL.
            if "/dentiste/" in href:
                liens.add(href.split("?")[0])

        if maximum and len(liens) >= maximum:
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
        hauteur = driver.execute_script("return document.body.scrollHeight;")
        if hauteur == hauteur_precedente:
            break
        hauteur_precedente = hauteur

    liens_tries = sorted(liens)
    return liens_tries[:maximum] if maximum else liens_tries


liens_profils = recuperer_liens_profils(driver, MAX_DENTISTES)
print(f"{len(liens_profils)} fiche(s) trouvée(s).")
liens_profils[:5]

## Étape 6 - Ouvrir chaque fiche et récupérer les informations

Pour chaque dentiste, Selenium ouvre sa fiche, clique sur le bouton à trois points de la présentation lorsqu'il existe, puis récupère la présentation complète.

In [ ]:
def extraire_fiche_dentiste(driver, url):
    driver.get(url)
    WebDriverWait(driver, TIMEOUT).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

    nom = texte_premier(driver, [
        (By.TAG_NAME, "h1"),
        (By.CSS_SELECTOR, "[data-test='doctor-name']")
    ])

    adresse = texte_premier(driver, [
        (By.CSS_SELECTOR, "[data-test='address']"),
        (By.XPATH, "//*[contains(@class, 'address')]"),
        (By.XPATH, "//*[contains(text(), 'Voir la carte')]/preceding::*[self::div or self::p][1]")
    ], valeur_par_defaut="Adresse non trouvée")

    # Le bouton peut être affiché sous la forme de trois points ou d'un libellé "Voir plus".
    attendre_et_cliquer(driver, [
        (By.XPATH, "//button[contains(@aria-label, 'Voir plus') or contains(@aria-label, 'présentation')]"),
        (By.XPATH, "//button[contains(normalize-space(.), 'Voir plus')]"),
        (By.XPATH, "//button[normalize-space(.)='…' or normalize-space(.)='...']")
    ], timeout=2)

    presentation = texte_premier(driver, [
        (By.CSS_SELECTOR, "[data-test='presentation']"),
        (By.XPATH, "//*[contains(@class, 'presentation')]"),
        (By.XPATH, "//*[contains(normalize-space(.), 'Présentation')]/following-sibling::*[1]")
    ], valeur_par_defaut="Présentation non trouvée")

    return {
        "nom": nom,
        "adresse": adresse,
        "presentation": presentation,
        "url": url
    }


dentistes = []
for index, url in enumerate(liens_profils, start=1):
    try:
        fiche = extraire_fiche_dentiste(driver, url)
        dentistes.append(fiche)
        print(f"[{index}/{len(liens_profils)}] {fiche['nom']}")
    except Exception as erreur:
        print(f"[{index}/{len(liens_profils)}] Fiche ignorée : {url} ({erreur})")

df_dentistes = pd.DataFrame(dentistes)
df_dentistes

## Étape 7 - Exporter les résultats et fermer le navigateur

In [ ]:
fichier_sortie = Path("dentistes_paris.csv")
df_dentistes.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")
print(f"Résultats enregistrés dans : {fichier_sortie.resolve()}")

driver.quit()